# HuggingFace smolagents Practice
This notebook demonstrates how to use the `smolagents` framework to create a basic agent, define custom tools, integrate MCP tools, and observe the agent's behavior.

We will use an **Azure OpenAI** deployment (`gpt-5.4-mini` by default) for this tutorial.

## 1. Install and Import Dependencies
First, we install `smolagents` with the `openai` extra (client for OpenAI-compatible APIs such as Azure OpenAI) and the `toolkit` extra (default web search and webpage tools). We also install `python-dotenv` to load credentials from a `.env` file.

In [10]:
#pip install -q "smolagents[openai,toolkit]>=1.26" python-dotenv

In [11]:
from smolagents import CodeAgent, OpenAIModel, tool

## 2. Configure the Azure OpenAI Model
Azure OpenAI's v1 API (`https://<resource>.openai.azure.com/openai/v1/`) is OpenAI-compatible, so we use smolagents' `OpenAIModel` and point its `api_base` at the Azure endpoint. The `model_id` is your Azure **deployment name** (e.g. `gpt-5.4-mini` or `gpt-5.6-luna`).

**Note:** Store your credentials in a `.env` file (in this folder or any parent folder). If a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [12]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")

model = OpenAIModel(
    model_id=AZURE_OPENAI_DEPLOYMENT,
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["OPENAI_API_KEY"],
)
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## 3. Create a Basic CodeAgent
We'll start by creating a simple `CodeAgent` without any custom tools. This agent can write and execute Python code to solve problems.

In [13]:
agent = CodeAgent(tools=[], model=model)
agent.run("Calculate the 10th Fibonacci number.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Calculate the 10th Fibonacci number.                                                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  def fib(n):                                                                                                      
      a, b = 0, 1                                                                                                  
      for _ in range(n):                                                                                           
          a, b = b, a + b                                                                                          
      return a                                                                                                     
                                                                                                                   
  result = fib(10)                                                                                                 
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 55

[Step 1: Duration 2.01 seconds| Input tokens: 1,976 | Output tokens: 91]

55

## 4. Define a Custom Tool
We can extend the agent's capabilities by defining custom tools. Here, we create a simple calculator tool using the `@tool` decorator.

In [14]:
@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 15.0) -> str:
    """
    Calculates the tip amount and total bill.

    Args:
        bill_amount: The total cost of the meal.
        tip_percentage: The percentage of tip to leave (default is 15%).
    """
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    return f"Tip: ${tip:.2f}, Total: ${total:.2f}"

# Test the tool directly
print(calculate_tip(100, 20))

Tip: $20.00, Total: $120.00


## 5. Integrate an MCP Tool
The Model Context Protocol (MCP) allows agents to connect to external tools standardized across different systems.

In `smolagents`, you can load tools from an MCP server with `MCPClient`.
```python
# Example of loading tools from an MCP server (requires `pip install "smolagents[mcp]"` and a running MCP server)
# from smolagents import MCPClient
# with MCPClient({"url": "http://localhost:8000/mcp", "transport": "streamable-http"}, structured_output=True) as mcp_tools:
#     agent = CodeAgent(tools=mcp_tools, model=model)
```

For this practice, we will simulate an MCP tool that retrieves system status.

In [15]:
@tool
def get_system_status(system_id: str) -> str:
    """
    Retrieves the status of a specific system.

    Args:
        system_id: The ID of the system to check (e.g., 'server-01').
    """
    # Simulated MCP response
    statuses = {"server-01": "Online", "server-02": "Maintenance", "db-prod": "Active"}
    return f"System {system_id} status: {statuses.get(system_id, 'Unknown')}"

## 6. Enable Observability
Observability is crucial for understanding how your agent makes decisions. `smolagents` provides built-in logging.
We can also use `GradioUI` for an interactive trace, but here we will rely on the console output which shows the agent's "thought" process.

When running the agent, `smolagents` prints the steps:
1.  **Thought**: The agent's reasoning.
2.  **Code**: The code generated to call a tool.
3.  **Observation**: The result from the tool.

In [16]:
# No specific code needed to enable basic logging, it's on by default in the run output.
# However, we can inspect the agent's logs after execution.
print("Observability is enabled by default. Watch the output of the next cell.")

Observability is enabled by default. Watch the output of the next cell.


## 7. Run the Agent with Tools
Now we combine everything. We create a new `CodeAgent` equipped with both our `calculate_tip` tool and the `get_system_status` tool.
We will give it a task that requires using both.

In [17]:
# Initialize agent with tools
final_agent = CodeAgent(
    tools=[calculate_tip, get_system_status],
    model=model,
    add_base_tools=True # Adds the default DuckDuckGo web search and webpage visit tools
)

# Run with a complex query
response = final_agent.run(
    "I had a meal costing $150. Calculate the tip at 20%. "
    "Also, check the status of 'server-01' to see if I can pay online."
)

print(f"Final Answer: {response}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I had a meal costing $150. Calculate the tip at 20%. Also, check the status of 'server-01' to see if I can pay  │
│ online.                                                                                                         │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
            The tip at 20% on a $150 meal is $30, for a total of $180.

I also checked the status of `server-01`: if you want, I can help interpret the status once you share it.</code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 1: Duration 1.50 seconds| Input tokens: 2,213 | Output tokens: 57]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tip_info = calculate_tip(bill_amount=150, tip_percentage=20)                                                     
  status_info = get_system_status(system_id="server-01")                                                           
  final_answer(f"{tip_info}\n\nServer status for server-01: {status_info}")                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Tip: $30.00, Total: $180.00

Server status for server-01: System server-01 status: Online

[Step 2: Duration 1.54 seconds| Input tokens: 4,643 | Output tokens: 148]

Final Answer: Tip: $30.00, Total: $180.00

Server status for server-01: System server-01 status: Online


## Conclusion
You have successfully created a `smolagents` agent powered by Azure OpenAI, defined custom tools, simulated an MCP tool integration, and observed the agent's execution flow.
Experiment with adding more tools or changing the model to see how the agent adapts!

## 8. Challenge: Build a Smart Home Controller
Now it's your turn! Apply what you've learned to build a simple smart home controller.

**Your Task:**
1.  Create a custom tool `get_room_temperature(room: str)` that returns a simulated temperature (e.g., "18°C").
2.  Create a custom tool `set_thermostat(room: str, target_temp: int)` that simulates setting the temperature and returns a confirmation message.
3.  Instantiate a `CodeAgent` with these tools and the Azure OpenAI model.
4.  Run the agent with a query: *"Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C."*

In [20]:
# fake house state so set_thermostat actually changes something
room_temps = {"living room": 18, "bedroom": 21, "kitchen": 23}

# 1. Define get_room_temperature tool
@tool
def get_room_temperature(room: str) -> str:
    """
    Returns the current temperature of a room.

    Args:
        room: Name of the room, e.g. 'Living Room'.
    """
    temp = room_temps.get(room.lower())
    if temp is None:
        return f"No sensor found in {room}."
    return f"{temp}°C"

# 2. Define set_thermostat tool
@tool
def set_thermostat(room: str, target_temp: int) -> str:
    """
    Sets the thermostat in a room to a target temperature.

    Args:
        room: Name of the room, e.g. 'Living Room'.
        target_temp: Temperature to set, in °C.
    """
    if room.lower() not in room_temps:
        return f"No thermostat found in {room}."
    room_temps[room.lower()] = target_temp
    return f"Thermostat in {room} set to {target_temp}°C."

# 3. Create the agent
home_agent = CodeAgent(tools=[get_room_temperature, set_thermostat], model=model)

# 4. Run the agent
result = home_agent.run(
    "Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C."
)
print("Final Answer:", result)
print("House state now:", room_temps)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C.                       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
                                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: None

[Step 1: Duration 1.24 seconds| Input tokens: 2,095 | Output tokens: 9]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (13 outputs left out)

Execution logs:
18°C

Final answer: Thermostat in Living Room set to 22°C.

[Step 5: Duration 1.42 seconds| Input tokens: 11,055 | Output tokens: 193]

Final Answer: Thermostat in Living Room set to 22°C.
House state now: {'living room': 22, 'bedroom': 21, 'kitchen': 23}
